In [ ]:
import pandas as pd

customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
category_translation = pd.read_csv(
    "../data/raw/product_category_name_translation.csv"
)

In [2]:
executive_df = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(items[["order_id","price"]], on="order_id", how="left")
)

executive_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,price
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,29.99
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,118.70
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,159.90
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,45.00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,19.90


In [3]:
executive_df.to_csv(
    "../data/processed/executive_dashboard.csv",
    index=False
)

In [4]:
sla_df = orders.copy()

sla_df["order_purchase_timestamp"] = pd.to_datetime(
    sla_df["order_purchase_timestamp"]
)

sla_df["order_delivered_customer_date"] = pd.to_datetime(
    sla_df["order_delivered_customer_date"]
)

sla_df["order_estimated_delivery_date"] = pd.to_datetime(
    sla_df["order_estimated_delivery_date"]
)

sla_df["delivery_days"] = (
    sla_df["order_delivered_customer_date"]
    -
    sla_df["order_purchase_timestamp"]
).dt.days

sla_df["is_late"] = (
    sla_df["order_delivered_customer_date"]
    >
    sla_df["order_estimated_delivery_date"]
).astype(int)

In [5]:
sla_df.to_csv(
    "../data/processed/sla_dashboard.csv",
    index=False
)

In [ ]:
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

orders = orders[
    orders["order_status"] == "delivered"
].copy()

orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

customer_orders = (
    orders
    .merge(
        customers[
            [
                "customer_id",
                "customer_unique_id"
            ]
        ],
        on="customer_id",
        how="left"
    )
    .merge(
        items[
            [
                "order_id",
                "price"
            ]
        ],
        on="order_id",
        how="left"
    )
)

snapshot_date = (
    customer_orders["order_purchase_timestamp"].max()
    + pd.Timedelta(days=1)
)

rfm = customer_orders.groupby(
    "customer_unique_id"
).agg({
    "order_purchase_timestamp":
        lambda x: (
            snapshot_date - x.max()
        ).days,
    "order_id":"nunique",
    "price":"sum"
})

rfm.columns = [
    "recency",
    "frequency",
    "monetary"
]

In [8]:
rfm["R_score"] = pd.qcut(
    rfm["recency"],
    5,
    labels=[5,4,3,2,1]
)

rfm["F_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

rfm["M_score"] = pd.qcut(
    rfm["monetary"],
    5,
    labels=[1,2,3,4,5]
)

rfm["RFM_Cell"] = (
    rfm["R_score"].astype(str)
    + rfm["F_score"].astype(str)
    + rfm["M_score"].astype(str)
)

def assign_segment(cell):
    if cell in ["555","554","545","455"]:
        return "VIP Customer"
    elif cell in ["111","112","121"]:
        return "At Risk"
    else:
        return "Regular Customer"

rfm["segment"] = rfm["RFM_Cell"].apply(assign_segment)

In [9]:
rfm.to_csv(
    "../data/processed/customer_dashboard.csv",
    index=False
)

In [10]:
revenue_df = (
    orders
    .merge(customers,on="customer_id")
    .merge(
        items[
            [
                "order_id",
                "product_id",
                "price"
            ]
        ],
        on="order_id"
    )
    .merge(
        products[
            [
                "product_id",
                "product_category_name"
            ]
        ],
        on="product_id",
        how="left"
    )
    .merge(
        category_translation,
        on="product_category_name",
        how="left"
    )
)

In [11]:
revenue_df.to_csv(
    "../data/processed/revenue_dashboard.csv",
    index=False
)

In [12]:
import os

os.listdir("../data/processed")

['customer_dashboard.csv',
 'executive_dashboard.csv',
 'revenue_dashboard.csv',
 'sla_dashboard.csv']